# DPI Probe Gate: adaptive-attack test (attack.py)

English injection scenarios only, synthetic data, dry-run. Paraphrases by Qwen3-8B; the attacker keeps the variant with the lowest probe score. Probes come from `results/<model>_probe.npz` in the code dataset.

In [ ]:
# The GitHub repo is private, so the code comes in as a private Kaggle dataset
# (dataset_sources in kernel-metadata.json, or Add Input in the UI).
import glob, os, shutil, zipfile
!nvidia-smi -L
W = "/kaggle/working/safl-hackathon"
hits = glob.glob("/kaggle/input/**/collect.py", recursive=True)
if hits:
    shutil.copytree(os.path.dirname(hits[0]), W, dirs_exist_ok=True)
else:
    zips = glob.glob("/kaggle/input/**/code.zip", recursive=True)
    assert zips, "code dataset not attached: nothing under /kaggle/input"
    zipfile.ZipFile(zips[0]).extractall(W)
%cd {W}
!ls -R | head -40

In [ ]:
# Gemma-4 (dhee-pravega) needs a recent transformers; 5.7.0 is the version checked locally.
!pip install -q "transformers==5.7.0" accelerate
import transformers, torch
print(transformers.__version__, torch.__version__, torch.cuda.device_count(), "GPUs")

In [ ]:
import os
try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN set from Kaggle secrets")
except Exception as e:
    print("No HF_TOKEN secret; public downloads only:", type(e).__name__)

In [ ]:
!mkdir -p /kaggle/working/logs && df -h / /kaggle/working | tail -2

In [ ]:
!ls results && python attack.py paraphrase 2>&1 | tee /kaggle/working/logs/attack_paraphrase.log

In [ ]:
!python attack.py run --model qwen3-8b --batch-size 1 2>&1 | tee /kaggle/working/logs/attack_run_qwen.log
!python attack.py run --model dhee-pravega --batch-size 8 2>&1 | tee /kaggle/working/logs/attack_run_dhee.log

In [ ]:
!rm -rf ~/.cache/huggingface/hub/models--Qwen--Qwen3-8B ~/.cache/huggingface/hub/models--dheeyantra--dhee-pravega; df -h / | tail -1

In [ ]:
!pip install -q bitsandbytes
!python attack.py judge --model qwen3-8b --load-4bit 2>&1 | tee /kaggle/working/logs/attack_judge_qwen.log
!python attack.py judge --model dhee-pravega --load-4bit 2>&1 | tee /kaggle/working/logs/attack_judge_dhee.log

In [ ]:
!zip -q -r /kaggle/working/attack_outputs.zip data/attack && ls -la /kaggle/working/attack_outputs.zip